# Preprocessing Data untuk Topic Modeling

**Tujuan**: Melakukan preprocessing teks yang mencakup stopword removal dan stemming untuk mendapatkan teks yang siap digunakan dalam topic modeling.

**Langkah-langkah:**
1. Load data hasil cleaning
2. Stopword Removal (menghapus kata umum)
3. Stemming (mengurangi kata ke akar bentuknya)
4. Visualisasi setiap langkah
5. Simpan hasil preprocessing

## 1. Setup dan Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import os
import warnings
warnings.filterwarnings('ignore')

# NLTK untuk stopword dan stemming
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Sastrawi untuk stemming Bahasa Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

print("Library berhasil diimpor!")

Library berhasil diimpor!


## 2. Load Data

In [2]:
# Cek apakah ada data_cleaned.csv, jika tidak gunakan hasil_processing.csv
if os.path.exists('data/data_cleaned.csv'):
    df = pd.read_csv('data/data_cleaned.csv')
    print("Load dari: data/data_cleaned.csv")
else:
    df = pd.read_csv('data/hasil_processing.csv')
    print("Load dari: data/hasil_processing.csv")

print(f"Shape: {df.shape}")
print(f"\nKolom: {df.columns.tolist()}")

# Ambil teks untuk preprocessing
if 'text_clean_advanced' in df.columns:
    texts = df['text_clean_advanced'].dropna().astype(str)
elif 'text_clean' in df.columns:
    texts = df['text_clean'].dropna().astype(str)
else:
    texts = df['text'].dropna().astype(str)

print(f"\nJumlah teks untuk preprocessing: {len(texts)}")

Load dari: data/data_cleaned.csv
Shape: (13537, 18)

Kolom: ['text', 'diggCount', 'replyCommentTotal', 'createTimeISO', 'uniqueId', 'videoWebUrl', 'source_file', 'text_clean', 'tokens', 'token_count', 'tokens_nostop', 'token_count_nostop', 'text_clean_advanced', 'text_length', 'word_count', 'date', 'month', 'hour']

Jumlah teks untuk preprocessing: 13537


## 3. Setup Stopwords dan Stemmer

In [3]:
# Indonesian Stopwords
stopwords_ind = set(stopwords.words('indonesian'))

# Tambah stopwords custom untuk konteks MBG
custom_stopwords = {
    'ya', 'yaa', 'yaaa', 'yg', 'yang', 'di', 'dari', 'dan', 'untuk', 'ini', 'itu',
    'dengan', 'pada', 'ke', 'adalah', 'juga', 'akan', 'atau', 'tersebut', 'bisa',
    'tidak', 'ada', 'sudah', 'saya', 'kita', 'mereka', 'dia', 'kami', 'kalo',
    'kalau', 'nih', 'nah', 'sih', 'dong', 'deh', 'dong', 'lho', 'loh', 'tuh',
    'pak', 'bu', 'bapak', 'ibu', 'mbg', 'bgn', 'ag', 'aja', 'saja', 'banget',
    'banget', 'sangat', 'sudah', 'masih', 'udah', 'akan', 'lagi', 'ttp', 'tetap',
    'jd', 'jadi', 'utk', 'utk', 'dgn', 'krn', 'karena', 'jd', 'jdi', 'sdh',
    'bs', 'bisa', 'ga', 'gak', 'nggak', 'nggk', 'gimana', 'gimana', ' Gimana',
    ' gimana', 'dong', 'dongs', 'kak', 'kakak', 'bang', 'teh', 'tuh', 'ni',
    'emang', 'emangnya', 'bener', 'betul', 'sy', 'aku', 'ku', 'mu', 'kau',
    'yaudah', 'ya', 'yaa', 'yok', 'ayo', 'yuk', 'gitu', 'begitu', 'gtu',
    'dm', 'dm', 'gpp', 'gakpapa', 'ok', 'oke', 'sip', 'sipp', 'siapp',
    'please', 'pls', 'tolong', 'tlg', 'thx', 'thanks', 'thank', 'you',
    'wkwk', 'wkwkwk', 'haha', 'hahaha', 'lol', 'lmao', 'btw', 'rt',
    'via', 'amp', 'the', 'and', 'of', 'to', 'is', 'in', 'for', 'on',
    'jd', 'jika', 'bila', 'apabila', 'bahwa', 'sejak', 'sejak', 'antara'
}

all_stopwords = stopwords_ind.union(custom_stopwords)

print(f"Jumlah stopwords Indonesia (default): {len(stopwords_ind)}")
print(f"Jumlah total stopwords (dengan custom): {len(all_stopwords)}")

# Setup Stemmer Sastrawi untuk Bahasa Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()

print("\nStemmer Sastrawi untuk Bahasa Indonesia siap!")

Jumlah stopwords Indonesia (default): 757
Jumlah total stopwords (dengan custom): 843


AttributeError: 'StemmerFactory' object has no attribute 'createStemmer'

## 4. Fungsi Preprocessing

In [ ]:
def preprocess_text(text, stemmer=None, stopwords_set=None, apply_stemming=True):
    """
    Fungsi preprocessing lengkap:
    1. Lowercase
    2. Remove special characters (opsional)
    3. Tokenization
    4. Stopword Removal
    5. Stemming
    """
    if pd.isna(text) or str(text).strip() == '':
        return []
    
    text = str(text).lower()
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove non-alphabetic tokens (opsional)
    tokens = [t for t in tokens if t.isalpha()]
    
    # Stopword Removal
    if stopwords_set:
        tokens = [t for t in tokens if t not in stopwords_set]
    
    # Stemming (hanya jika ada stemmer)
    if apply_stemming and stemmer:
        tokens = [stemmer.stem(t) for t in tokens]
    
    return tokens

def tokens_to_string(tokens):
    """Convert tokens back to string"""
    return ' '.join(tokens)

print("Fungsi preprocessing siap!")

## 5. STOPWORD REMOVAL

In [ ]:
# Visualisasi: Sebelum Stopword Removal
print("=" * 60)
print("SEBELUM STOPWORD REMOVAL")
print("=" * 60)

# Hitung frekuensi kata sebelum stopword removal
all_words_before = []
for text in texts.head(5000):  # Sample untuk efisiensi
    tokens = word_tokenize(str(text).lower())
    all_words_before.extend([t for t in tokens if t.isalpha()])

word_freq_before = Counter(all_words_before)
top_20_before = word_freq_before.most_common(20)

print("\nTop 20 kata sebelum stopword removal:")
for word, count in top_20_before:
    print(f"  {word}: {count}")

# Visualisasi
fig, ax = plt.subplots(figsize=(12, 6))
words, counts = zip(*top_20_before)
sns.barplot(x=list(counts), y=list(words), palette='Reds_r', ax=ax)
ax.set_title('Top 20 Kata SEBELUM Stopword Removal', fontsize=14, fontweight='bold')
ax.set_xlabel('Frekuensi')
ax.set_ylabel('Kata')
plt.tight_layout()
plt.savefig('output/step1_before_stopword.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Apply Stopword Removal
print("=" * 60)
print("STOPWORD REMOVAL")
print("=" * 60)

texts_nostop = []
tokens_nostop_list = []

for text in texts:
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha() and t not in all_stopwords]
    tokens_nostop_list.append(tokens)
    texts_nostop.append(' '.join(tokens))

print(f"Jumlah teks setelah stopword removal: {len(texts_nostop)}")

# Hitung frekuensi setelah stopword removal
all_words_after_stop = [w for tokens in tokens_nostop_list for w in tokens]
word_freq_after_stop = Counter(all_words_after_stop)
top_20_after_stop = word_freq_after_stop.most_common(20)

print("\nTop 20 kata setelah stopword removal:")
for word, count in top_20_after_stop:
    print(f"  {word}: {count}")

In [ ]:
# Visualisasi: Setelah Stopword Removal
fig, ax = plt.subplots(figsize=(12, 6))
words, counts = zip(*top_20_after_stop)
sns.barplot(x=list(counts), y=list(words), palette='Greens_r', ax=ax)
ax.set_title('Top 20 Kata SETELAH Stopword Removal', fontsize=14, fontweight='bold')
ax.set_xlabel('Frekuensi')
ax.set_ylabel('Kata')
plt.tight_layout()
plt.savefig('output/step2_after_stopword.png', dpi=150, bbox_inches='tight')
plt.show()

# Perbandingan
print("\n" + "=" * 60)
print("PERBANDINGAN STOPWORD REMOVAL")
print("=" * 60)
print(f"Total kata sebelum: {len(all_words_before):,}")
print(f"Total kata setelah: {len(all_words_after_stop):,}")
print(f"Pengurangan: {len(all_words_before) - len(all_words_after_stop):,} kata ({(len(all_words_before) - len(all_words_after_stop))/len(all_words_before)*100:.1f}%)")

In [ ]:
# Contoh hasil stopword removal
print("\nContoh hasil Stopword Removal:")
print("-" * 60)
for i in range(3):
    print(f"\nSebelum: {texts.iloc[i][:200]}")
    print(f"Sesudah: {' '.join(tokens_nostop_list[i][:20])}...")

## 6. STEMMING

In [ ]:
print("=" * 60)
print("STEMMING")
print("=" * 60)

# Apply Stemming
texts_stemmed = []
tokens_stemmed_list = []

print("Mulai stemming... (ini mungkin memakan waktu)")

for i, tokens in enumerate(tokens_nostop_list):
    stemmed_tokens = []
    for token in tokens:
        try:
            stemmed = stemmer.stem(token)
            stemmed_tokens.append(stemmed)
        except:
            stemmed_tokens.append(token)
    
    tokens_stemmed_list.append(stemmed_tokens)
    texts_stemmed.append(' '.join(stemmed_tokens))
    
    # Progress indicator
    if (i + 1) % 2000 == 0:
        print(f"  Processed: {i+1}/{len(tokens_nostop_list)}")

print(f"\nStemming selesai! {len(texts_stemmed)} teks diproses.")

In [ ]:
# Hitung frekuensi setelah stemming
all_words_after_stem = [w for tokens in tokens_stemmed_list for w in tokens]
word_freq_after_stem = Counter(all_words_after_stem)
top_20_after_stem = word_freq_after_stem.most_common(20)

print("\nTop 20 kata setelah stemming:")
for word, count in top_20_after_stem:
    print(f"  {word}: {count}")

# Visualisasi
fig, ax = plt.subplots(figsize=(12, 6))
words, counts = zip(*top_20_after_stem)
sns.barplot(x=list(counts), y=list(words), palette='Blues_r', ax=ax)
ax.set_title('Top 20 Kata SETELAH Stemming', fontsize=14, fontweight='bold')
ax.set_xlabel('Frekuensi')
ax.set_ylabel('Kata')
plt.tight_layout()
plt.savefig('output/step3_after_stemming.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Contoh hasil stemming
print("\nContoh hasil Stemming:")
print("-" * 60)
for i in range(3):
    print(f"\nSebelum (no stopword): {' '.join(tokens_nostop_list[i][:15])}")
    print(f"Setelah stemming:       {' '.join(tokens_stemmed_list[i][:15])}")

## 7. PERBANDINGAN KESELURUHAN

In [ ]:
# Visualisasi perbandingan 3 tahap
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# Step 1: Sebelum preprocessing
words1, counts1 = zip(*top_20_before)
sns.barplot(x=list(counts1), y=list(words1), palette='Reds_r', ax=axes[0])
axes[0].set_title('SEBELUM Preprocessing', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Frekuensi')

# Step 2: Setelah stopword removal
words2, counts2 = zip(*top_20_after_stop)
sns.barplot(x=list(counts2), y=list(words2), palette='Greens_r', ax=axes[1])
axes[1].set_title('SETELAH Stopword Removal', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Frekuensi')

# Step 3: Setelah stemming
words3, counts3 = zip(*top_20_after_stem)
sns.barplot(x=list(counts3), y=list(words3), palette='Blues_r', ax=axes[2])
axes[2].set_title('SETELAH Stemming', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Frekuensi')

plt.suptitle('Perbandingan Top 20 Kata pada Setiap Tahap Preprocessing', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/preprocessing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics
print("=" * 60)
print("SUMMARY PREPROCESSING")
print("=" * 60)

print(f"\nJumlah teks:")
print(f"  - Teks asli: {len(texts)}")
print(f"  - Setelah stopword removal: {len(texts_nostop)}")
print(f"  - Setelah stemming: {len(texts_stemmed)}")

print(f"\nJumlah kata unik:")
print(f"  - Sebelum preprocessing: {len(set(all_words_before)):,}")
print(f"  - Setelah stopword removal: {len(set(all_words_after_stop)):,}")
print(f"  - Setelah stemming: {len(set(all_words_after_stem)):,}")

print(f"\nTotal kata:")
print(f"  - Sebelum preprocessing: {len(all_words_before):,}")
print(f"  - Setelah stopword removal: {len(all_words_after_stop):,}")
print(f"  - Setelah stemming: {len(all_words_after_stem):,}")

## 8. VISUALISASI WORDCLOUD

In [ ]:
from wordcloud import WordCloud

# WordCloud setelah preprocessing
text_final = ' '.join(all_words_after_stem)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# WordCloud hasil akhir
wordcloud = WordCloud(
    width=800, height=400,
    background_color='white',
    colormap='viridis',
    max_words=100,
    min_font_size=10
).generate(text_final)

axes[0].imshow(wordcloud, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('WordCloud Setelah Preprocessing', fontsize=14, fontweight='bold')

# WordCloud hanya high-frequency words
wordcloud2 = WordCloud(
    width=800, height=400,
    background_color='black',
    colormap='plasma',
    max_words=50,
    min_font_size=12
).generate_from_frequencies(dict(word_freq_after_stem.most_common(100)))

axes[1].imshow(wordcloud2, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Top 100 Kata (Frequency-based)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('output/wordcloud_final.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. DISTRIBUSI PANJANG TEKS

In [ ]:
# Hitung panjang teks di setiap tahap
len_before = [len(word_tokenize(str(t).lower())) for t in texts]
len_after_stop = [len(t) for t in tokens_nostop_list]
len_after_stem = [len(t) for t in tokens_stemmed_list]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(len_before, bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribusi Panjang Teks
SEBELUM', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Jumlah Kata')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(np.mean(len_before), color='red', linestyle='--', label=f'Mean: {np.mean(len_before):.1f}')
axes[0].legend()

axes[1].hist(len_after_stop, bins=30, color='green', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribusi Panjang Teks
SETELAH Stopword Removal', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Jumlah Kata')
axes[1].set_ylabel('Frekuensi')
axes[1].axvline(np.mean(len_after_stop), color='red', linestyle='--', label=f'Mean: {np.mean(len_after_stop):.1f}')
axes[1].legend()

axes[2].hist(len_after_stem, bins=30, color='blue', edgecolor='black', alpha=0.7)
axes[2].set_title('Distribusi Panjang Teks
SETELAH Stemming', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Jumlah Kata')
axes[2].set_ylabel('Frekuensi')
axes[2].axvline(np.mean(len_after_stem), color='red', linestyle='--', label=f'Mean: {np.mean(len_after_stem):.1f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('output/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRingkasan Panjang Teks:")
print(f"  Sebelum preprocessing - Mean: {np.mean(len_before):.2f}, Median: {np.median(len_before):.2f}")
print(f"  Setelah stopword - Mean: {np.mean(len_after_stop):.2f}, Median: {np.median(len_after_stop):.2f}")
print(f"  Setelah stemming - Mean: {np.mean(len_after_stem):.2f}, Median: {np.median(len_after_stem):.2f}")

## 10. SIMPAN HASIL PREPROCESSING

In [ ]:
# Buat DataFrame hasil preprocessing
df_preprocessed = pd.DataFrame({
    'text_original': texts.values,
    'tokens_no_stop': [' '.join(t) for t in tokens_nostop_list],
    'tokens_stemmed': texts_stemmed,
    'token_count_before': len_before,
    'token_count_after_stop': len_after_stop,
    'token_count_after_stem': len_after_stem
})

# Merge dengan data original jika ada
if 'diggCount' in df.columns:
    df_preprocessed['diggCount'] = df['diggCount'].values[:len(texts)]
if 'uniqueId' in df.columns:
    df_preprocessed['uniqueId'] = df['uniqueId'].values[:len(texts)]
if 'createTimeISO' in df.columns:
    df_preprocessed['createTimeISO'] = df['createTimeISO'].values[:len(texts)]

# Simpan hasil
df_preprocessed.to_csv('data/data_preprocessed.csv', index=False)
print(f"Hasil preprocessing disimpan ke: data/data_preprocessed.csv")
print(f"Shape: {df_preprocessed.shape}")

print("\nPreview hasil:")
df_preprocessed.head()

In [ ]:
# Simpan juga versi yang siap untuk Topic Modeling (bag-of-words)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# CountVectorizer untuk LDA
count_vectorizer = CountVectorizer(
    max_df=0.95,  # Abaikan kata yang muncul di >95% dokumen
    min_df=5,     # Abaikan kata yang muncul di <5 dokumen
    max_features=5000  # Batasi 5000 kata teratas
)

count_matrix = count_vectorizer.fit_transform(df_preprocessed['tokens_stemmed'])
count_df = pd.DataFrame(
    count_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out()
)

count_df.to_csv('data/bow_countvectorizer.csv', index=False)
print(f"Bag-of-Words (CountVectorizer) disimpan ke: data/bow_countvectorizer.csv")
print(f"Shape: {count_df.shape}")

# TF-IDF untuk NMF
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=5,
    max_features=5000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df_preprocessed['tokens_stemmed'])
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

tfidf_df.to_csv('data/tfidf_vectorizer.csv', index=False)
print(f"TF-IDF Vectorizer disimpan ke: data/tfidf_vectorizer.csv")
print(f"Shape: {tfidf_df.shape}")

## 11. SUMMARY

In [ ]:
print("=" * 60)
print("PREPROCESSING COMPLETE!")
print("=" * 60)

print("\n📊 HASIL:")
print(f"   Total dokumen: {len(df_preprocessed):,}")
print(f"   Total fitur (kata): {count_df.shape[1]:,}")

print("\n📁 FILE YANG DISIMPAN:")
print("   1. data/data_preprocessed.csv - Teks yang sudah diproses")
print("   2. data/bow_countvectorizer.csv - Matriks BoW untuk LDA")
print("   3. data/tfidf_vectorizer.csv - Matriks TF-IDF untuk NMF")

print("\n📈 VISUALISASI:")
print("   1. output/step1_before_stopword.png")
print("   2. output/step2_after_stopword.png")
print("   3. output/step3_after_stemming.png")
print("   4. output/preprocessing_comparison.png")
print("   5. output/wordcloud_final.png")
print("   6. output/text_length_distribution.png")

print("\n✅ Data siap untuk Topic Modeling!")

---

## Langkah Berikutnya

Setelah preprocessing selesai, lanjut ke:
1. **Topic Modeling** - LDA atau NMF
2. **Visualisasi Interaktif** - pyLDAvis
3. **Interpretasi Hasil**

File notebook: `topic_modeling.ipynb`